# Portfolio returns and multi-period attribution

This notebook covers holdings-based performance measurement and attribution exposed by `finstack_quant.portfolio`:

- **Time-weighted return (TWRR)** via Modified Dietz per sub-period, then geometric linking.
- **Money-weighted return (MWRR)** via XIRR on dated cashflows.
- **Brinson-Fachler attribution** -- splitting active return into allocation, selection, and interaction by sector for a single period.
- **Carino linking** -- combining single-period Brinson effects into multi-period contributions that reconcile to the geometric active return.

Specs go in as JSON, and results come back as typed wrappers (`LinkedReturn`, `BrinsonPeriodResult`, `CarinoLinkedAttribution`) with getters and `to_pandas()` exits. Use `to_json()` to obtain the canonical wire string for logging and replay.

## Imports

In [1]:
import json

import pandas as pd

from finstack_quant.portfolio import (
    twrr_modified_dietz,
    twrr_linked,
    mwr_xirr,
    brinson_fachler,
    carino_link,
)

## Time-weighted return (Modified Dietz per sub-period)

Each sub-period is a beginning market value (BMV), an ending market value (EMV), and external cashflows. For TWRR the **sign convention is portfolio-centric**: a contribution into the portfolio is **positive**, a withdrawal is **negative**. Each flow carries `fraction_of_period_remaining` -- the Dietz time weight in `[0, 1]` (a start-of-period flow is `1.0`, an end-of-period flow is `0.0`).

The Modified Dietz return is `(EMV - BMV - sum(CF)) / (BMV + sum(w * CF))`.

In [14]:
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/portfolio_returns_and_attribution.json").read_text())

sub_periods = _NOTEBOOK_DATA['sub_periods']

print(sub_periods)

sub_returns = [twrr_modified_dietz(p) for p in sub_periods]
pd.DataFrame({"sub_period": ["Q1", "Q2"], "modified_dietz_return": sub_returns})

[{'beginning_market_value': 10000000.0, 'cashflows': [{'amount': 1000000.0, 'fraction_of_period_remaining': 0.6}], 'ending_market_value': 10500000.0}, {'beginning_market_value': 10500000.0, 'cashflows': [{'amount': -500000.0, 'fraction_of_period_remaining': 0.4}], 'ending_market_value': 10150000.0}]


,sub_period,modified_dietz_return
0,Q1,-0.047170
1,Q2,0.014563


### Linking sub-period returns

`twrr_linked` compounds the sub-period returns geometrically (`prod(1 + r) - 1`). `horizon_years` is the elapsed time as a fraction of a 365-day year; annualisation only applies once the horizon reaches a full year.

In [3]:
half_year = twrr_linked(sub_returns, 0.5)
print("Two quarters (horizon 0.5y):")
print(f"  cumulative = {half_year.cumulative:.4%}   annualised = {half_year.annualised:.4%}   periods = {half_year.num_periods}")

# Over a longer horizon, annualisation differs from the cumulative figure
six_quarters = [0.02, 0.015, -0.01, 0.03, 0.012, 0.018]
linked = twrr_linked(six_quarters, 1.5)
print("\nSix quarters (horizon 1.5y):")
print(f"  cumulative = {linked.cumulative:.4%}   annualised = {linked.annualised:.4%}   periods = {linked.num_periods}")

Two quarters (horizon 0.5y):
  cumulative = -3.3294%   annualised = -3.3294%   periods = 2

Six quarters (horizon 1.5y):
  cumulative = 8.7594%   annualised = 5.7575%   periods = 6


## Money-weighted return (XIRR)

MWRR is the internal rate of return that prices the dated external cashflows to zero. Here the **sign convention is investor-centric and the opposite of TWRR**: money paid *into* the portfolio is **negative**, terminal value and distributions received are **positive**.

In [15]:
cashflows = [
    {"date": "2025-01-01", "amount": -10_000_000.0},  # initial investment
    {"date": "2025-04-01", "amount": -1_000_000.0},   # contribution
    {"date": "2025-10-01", "amount": 500_000.0},      # distribution
    {"date": "2026-01-01", "amount": 11_200_000.0},   # terminal value
]

irr = mwr_xirr(cashflows)
print(f"Money-weighted return (annualised XIRR): {irr:.4%}")

Money-weighted return (annualised XIRR): 6.5886%


TWRR strips out the effect of cashflow timing (it measures the manager's decisions), while MWRR reflects the investor's actual dollar-weighted experience. The two diverge whenever flows are large and poorly / well timed relative to performance.

## Brinson-Fachler attribution (single period)

Given portfolio and benchmark weights and returns per sector, Brinson-Fachler decomposes the active return into three effects per sector:

- **Allocation** $= (w_p - w_b)(r_b^{i} - r_b)$
- **Selection** $= w_b (r_p^{i} - r_b^{i})$
- **Interaction** $= (w_p - w_b)(r_p^{i} - r_b^{i})$

Weights must sum to 1.0 on each side.

In [5]:
def sector(name, wp, wb, rp, rb):
    return {
        "sector": name,
        "portfolio_weight": wp,
        "benchmark_weight": wb,
        "portfolio_return": rp,
        "benchmark_return": rb,
    }

period_1 = [
    sector("Tech", 0.45, 0.40, 0.085, 0.060),
    sector("Energy", 0.30, 0.35, 0.010, 0.030),
    sector("Health", 0.25, 0.25, 0.040, 0.035),
]

bf = brinson_fachler(period_1)
bf.to_pandas()

,sector,allocation,selection,interaction,total
0,Tech,0.000837,0.01000,0.00125,0.012088
1,Energy,0.000662,-0.00700,0.00100,-0.005337
2,Health,-0.000000,0.00125,0.00000,0.001250


In [6]:
summary = {
    "total_allocation": bf.total_allocation,
    "total_selection": bf.total_selection,
    "total_interaction": bf.total_interaction,
    "total_excess_return": bf.total_excess_return,
    "portfolio_return": bf.portfolio_return,
    "benchmark_return": bf.benchmark_return,
}
recon = bf.total_allocation + bf.total_selection + bf.total_interaction
print(f"allocation + selection + interaction = {recon:.4%}")
print(f"total excess return                  = {bf.total_excess_return:.4%}")
pd.Series(summary)

allocation + selection + interaction = 0.8000%
total excess return                  = 0.8000%


total_allocation       0.00150
total_selection        0.00425
total_interaction      0.00225
total_excess_return    0.00800
portfolio_return       0.05125
benchmark_return       0.04325
dtype: float64

## Multi-period attribution with Carino linking

Single-period effects do not add up arithmetically across periods (returns compound). Carino linking applies a smoothing coefficient per period so that the linked allocation / selection / interaction effects sum exactly to the **geometric** active return over the full horizon.

In [7]:
period_2 = [
    sector("Tech", 0.50, 0.40, 0.020, 0.030),
    sector("Energy", 0.25, 0.35, -0.010, 0.000),
    sector("Health", 0.25, 0.25, 0.015, 0.010),
]
period_3 = [
    sector("Tech", 0.40, 0.40, 0.030, 0.025),
    sector("Energy", 0.35, 0.35, 0.020, 0.015),
    sector("Health", 0.25, 0.25, 0.010, 0.012),
]

linked_attr = carino_link([brinson_fachler(p) for p in (period_1, period_2, period_3)])
linked_attr.to_pandas()

,sector,allocation,selection,interaction,total
0,Tech,0.002521,0.008179,0.000223,0.010923
1,Energy,0.002233,-0.009113,0.002101,-0.004779
2,Health,0.000000,0.002096,0.000000,0.002096


In [8]:
geometric_active = linked_attr.portfolio_return_compounded - linked_attr.benchmark_return_compounded
recon = (
    linked_attr.linked_allocation
    + linked_attr.linked_selection
    + linked_attr.linked_interaction
)
print(f"compounded portfolio return : {linked_attr.portfolio_return_compounded:.4%}")
print(f"compounded benchmark return : {linked_attr.benchmark_return_compounded:.4%}")
print(f"geometric active return     : {geometric_active:.4%}")
print(f"sum of linked effects       : {recon:.4%}")
print(f"reconciles: {abs(recon - geometric_active) < 1e-9}")

compounded portfolio return : 8.5933%
compounded benchmark return : 7.7693%
geometric active return     : 0.8240%
sum of linked effects       : 0.8240%
reconciles: True


## Takeaways

- `twrr_modified_dietz` computes a single sub-period TWRR (contributions positive); `twrr_linked` compounds sub-period returns and annualises once the horizon reaches a year.
- `mwr_xirr` computes the money-weighted (dollar-weighted) return; its cashflow signs are the opposite of the TWRR convention (money in is negative).
- `brinson_fachler` splits a single period's active return into allocation, selection, and interaction by sector, reconciling exactly to the total excess return.
- `carino_link` chains per-period `brinson_fachler` results so the linked effects reconcile to the geometric active return across the whole horizon.

## Analyst program: three periods from a natively repriced bond sleeve

Opening USD market values determine sleeve weights. Coupon receipts use payment-date FX;
EUR bond returns include FX. The benchmark rebalances to 50/50 in the same two instruments,
so active selection is zero by design. The compound return is a linked total-return index;
the whole book, its trades, and settled cash ledger are reconciled separately in lesson 3.1.

Use the dated Campisi example below to separate coupon carry from matched Treasury and spread effects.


In [9]:
import json
from datetime import date
import pandas as pd
from _shared.analyst_book import book_spec
from _shared.analyst_history import history_market, history_quotes, performance_history
from finstack_quant.portfolio import (
    Portfolio, value_portfolio, aggregate_full_cashflows, brinson_fachler, carino_link,
)
AS_OF = date(2025, 1, 15)

# The bond sleeve is fully invested and has positive capital. Its USD returns include FX.
# The whole book's flow-neutral performance is reconciled separately in lesson 3.1.
history_periods = performance_history()
history_dates = (date.fromisoformat(history_periods[0]['start']), *(date.fromisoformat(period['end']) for period in history_periods))
ids = ['USD-CORP', 'EUR-GOVT']
initial = book_spec('base')
initial['positions'] = [p for p in initial['positions'] if p['instrument_id'] in ids]
initial_book = Portfolio.from_spec(json.dumps(initial))
cashflows = aggregate_full_cashflows(initial_book, history_market(AS_OF)).events
brinson_periods, rows = [], []
for start, end in zip(history_dates[:-1], history_dates[1:], strict=True):
    start_spec, end_spec = dict(initial, as_of=start.isoformat()), dict(initial, as_of=end.isoformat())
    start_market, end_market = history_market(start), history_market(end)
    begin = value_portfolio(Portfolio.from_spec(json.dumps(start_spec)), start_market, metrics=[])
    finish = value_portfolio(Portfolio.from_spec(json.dumps(end_spec)), end_market, metrics=[])
    beginning = {iid: begin.position_values[iid].value_base.amount for iid in ids}
    ending = {iid: finish.position_values[iid].value_base.amount for iid in ids}
    receipts = dict.fromkeys(ids, 0.0)
    for event in cashflows:
        when = date.fromisoformat(event['date'])
        if start < when <= end and event['position_id'] in ids:
            fx = history_quotes(when)['eur_usd'] if event['amount']['currency']=='EUR' else 1.0
            receipts[event['position_id']] += float(event['amount']['amount']) * fx
    returns_by_id = {iid: (ending[iid]+receipts[iid])/beginning[iid]-1 for iid in ids}
    total_begin = sum(beginning.values())
    weights = {iid: beginning[iid]/total_begin for iid in ids}
    # An explicit 50/50 rebalanced benchmark holds the same two sector instruments.
    # Selection is consequently zero; active return measures allocation alone.
    period = [{'sector': iid, 'portfolio_weight': weights[iid], 'benchmark_weight': 0.5,
               'portfolio_return': returns_by_id[iid], 'benchmark_return': returns_by_id[iid]} for iid in ids]
    bf = brinson_fachler(period)
    assert abs(bf.total_allocation+bf.total_selection+bf.total_interaction-bf.total_excess_return) < 1e-12
    assert abs(bf.total_selection) < 1e-12 and abs(bf.total_interaction) < 1e-12
    pnl = sum(ending.values()) + sum(receipts.values()) - total_begin
    assert abs(pnl - total_begin * bf.portfolio_return) < 1e-8, 'USD sleeve instrument P&L identity'
    brinson_periods.append(bf)
    for iid in ids:
        rows.append({'period_end': end, 'position': iid, 'opening_usd': beginning[iid],
                     'closing_usd': ending[iid], 'receipts_usd': receipts[iid],
                     'total_return': returns_by_id[iid]})
linked = carino_link(brinson_periods)
active = linked.portfolio_return_compounded-linked.benchmark_return_compounded
assert abs(linked.linked_allocation+linked.linked_selection+linked.linked_interaction-active)<1e-12
print(pd.DataFrame(rows).to_string(index=False))
print('Carino linked active return:',active)
coupon_period = next(row for row in rows if row['receipts_usd']>0)
omitted_income_return = coupon_period['closing_usd']/coupon_period['opening_usd']-1
assert abs(coupon_period['total_return']-omitted_income_return-coupon_period['receipts_usd']/coupon_period['opening_usd'])<1e-12
print('Omitted coupon error equals coupon divided by the opening mark.')


period_end position  opening_usd  closing_usd  receipts_usd  total_return
2025-04-15 USD-CORP 2.086986e+06 2.130417e+06       0.00000      0.020811
2025-04-15 EUR-GOVT 1.618366e+06 1.616763e+06       0.00000     -0.000991
2025-07-16 USD-CORP 2.130417e+06 2.068410e+06   55000.00000     -0.003289
2025-07-16 EUR-GOVT 1.616763e+06 1.649491e+06   26804.55163      0.036822
2025-10-15 USD-CORP 2.068410e+06 2.094720e+06       0.00000      0.012720
2025-10-15 EUR-GOVT 1.649491e+06 1.648368e+06       0.00000     -0.000681
Carino linked active return: -0.0006125706468989023
Omitted coupon error equals coupon divided by the opening mark.


### Campisi return attribution with matched opening sensitivities

This controlled USD example uses independently specified opening coupon income, market values, and matched Treasury and settlement Z-spread experiments. The 73-day window gives a coupon fraction of 73/365. The rates and spread effects use the full closing quote move. The residual captures the part of realized return left after carry and these two price effects.

In [10]:
from finstack_quant.core.money import Money
from finstack_quant.portfolio import (
    FiAttributionConfig, FiAttributionResult, FiPositionSnapshot,
    FiQuote, FiQuoteLeg, FiReportingBasis, campisi_attribution, campisi_carino_link,
)

reporting = FiReportingBasis("portfolio", "benchmark", "USD total return", "dirty settlement marks", "USD native", "net of fees")
government_basis = "UST 5Y par yield, ACT/365F, settlement"
spread_basis = "USD-OIS Z-spread, ACT/365F, maturity workout"
treasury = FiQuoteLeg(
    government_basis, 5.0,
    FiQuote(government_basis, 0.03), FiQuote(government_basis, 0.031),
)
z_spread = FiQuoteLeg(spread_basis, 4.0, FiQuote(spread_basis, -0.0015), FiQuote(spread_basis, 0.001))
credit = FiPositionSnapshot(
    "A-CREDIT", "Credit", "1", 0.02, Money(80, "USD"), Money(5, "USD")
).with_quote_legs(treasury=treasury, spread=z_spread)
cash = FiPositionSnapshot("CASH", "Cash", "1", 0.0, Money(80, "USD"), Money(0, "USD"))
config = FiAttributionConfig(date(2025, 1, 1), date(2025, 3, 15), "USD", reporting, ["Credit", "Cash"])

attributed = campisi_attribution([credit], [cash], config)
split = attributed.portfolio_components
expected = {"carry": 0.0125, "treasury": -0.005, "spread": -0.01, "residual": 0.0225}
for component, value in expected.items():
    assert abs(split[component] - value) < 1e-12
assert attributed.reconciliation_check(1e-12).is_reconciled
second_config = FiAttributionConfig(date(2025, 3, 15), date(2025, 5, 27), "USD", reporting, ["Credit", "Cash"])
second = campisi_attribution([credit], [cash], second_config)
campisi_linked = campisi_carino_link([attributed, second])
assert abs(campisi_linked.active_return_compounded - (1.02**2 - 1)) < 1e-12
display(attributed.to_pandas())

# A restored result must still match the recorded source observations.
altered = json.loads(attributed.to_json())
altered["total_active_carry"] += 0.001
try:
    FiAttributionResult.from_json(json.dumps(altered))
except ValueError:
    pass
else:
    raise AssertionError("Altered report should fail source-backed reconciliation")

,sector,portfolio_weight,benchmark_weight,portfolio_return,benchmark_return,benchmark_comparator,allocation,active_carry,active_treasury,active_spread,residual,total_active
0,Credit,1.0,0.0,0.02,NaN,0.0,0.0,0.0125,-0.005,-0.01,0.0225,0.02
1,Cash,0.0,1.0,NaN,0.0,0.0,0.0,0.0000,0.000,0.00,0.0000,0.00


## A native whole-book monthly P&L waterfall


In [11]:
from _shared.analyst_book import instruments
from finstack_quant.valuations.instruments import price_instrument
from finstack_quant.attribution import attribute_pnl, default_waterfall_order
from finstack_quant.reporting.charts import waterfall_chart
from finstack_quant.reporting.theme import INSTITUTIONAL
from pathlib import Path
# Avoid coupon-date boundary ambiguity: this month has no external flows, trades or payments.
a0,a1=date(2025,1,16),date(2025,2,16)
m0,m1=history_market(a0),history_market(a1)
fields=["carry","rates_curves_pnl","credit_curves_pnl","inflation_curves_pnl","correlations_pnl","fx_pnl","vol_pnl","cross_factor_pnl","model_params_pnl","market_scalars_pnl","residual"]
waterfall_totals=dict.fromkeys(fields,0.0);waterfall_totals["reporting_fx_translation"]=0.0
instrument_rows=[]
for iid,env in instruments("base").items():
 result=attribute_pnl(
     json.dumps(env),
     {
         "market_t0": m0,
         "market_t1": m1,
         "as_of_t0": a0,
         "as_of_t1": a1,
         "method": {"waterfall":default_waterfall_order()},
     },
 )
 assert not result.result_invalid
 p0=price_instrument(json.dumps(env),m0,a0).value.amount
 p1=price_instrument(json.dumps(env),m1,a1).value.amount
 currency=result.currency
 fx0=history_quotes(a0)["eur_usd"] if currency=="EUR" else 1.0
 fx1=history_quotes(a1)["eur_usd"] if currency=="EUR" else 1.0
 assert abs(result.total_pnl-(p1-p0))<1e-6, (iid,"unexpected payment or date boundary")
 assert abs(sum(getattr(result, k) for k in fields)-result.total_pnl)<1e-6
 for k in fields: waterfall_totals[k]+=getattr(result, k)*fx1
 translation=p0*(fx1-fx0);waterfall_totals["reporting_fx_translation"]+=translation
 instrument_rows.append({"position":iid,"currency":currency,"native_pnl":result.total_pnl,"reporting_fx_usd":translation,"usd_pnl":p1*fx1-p0*fx0})
actual_usd=sum(row["usd_pnl"] for row in instrument_rows)
assert abs(sum(waterfall_totals.values())-actual_usd)<1e-6
print(pd.DataFrame(instrument_rows).to_string(index=False))
print("USD factor waterfall:",waterfall_totals)
Path("book-pnl-waterfall.svg").write_text(waterfall_chart(list(waterfall_totals),list(waterfall_totals.values()),theme=INSTITUTIONAL))
taylor=attribute_pnl(
    json.dumps(instruments("base")["USD-CORP"]),
    {
        "market_t0": m0,
        "market_t1": m1,
        "as_of_t0": a0,
        "as_of_t1": a1,
        "method": {"taylor":{"include_gamma":True,"rate_bump_bp":1.0,"credit_spread_bump_bp":1.0,"vol_bump":.01}},
    },
)
assert abs(taylor.total_pnl-next(r["native_pnl"] for r in instrument_rows if r["position"]=="USD-CORP"))<1e-6
print("USD bond Taylor residual, USD:",taylor.residual)
print("Taylor residual is disclosed approximation error, not added to explained factors as skill.")


     position currency   native_pnl  reporting_fx_usd      usd_pnl
     USD-CORP      USD 15302.528829          0.000000 15302.528829
     EUR-GOVT      EUR  4292.379317      -5161.932673  -541.424804
     USD-CASH      USD  2295.802240          0.000000  2295.802240
     EUR-CASH      EUR   645.630053       -861.217074  -166.232190
USD-PAYER-IRS      USD  1041.247048          0.000000  1041.247048
     SPX-CALL      USD   413.315079          0.000000   413.315079
     ACME-CDS      USD  2563.346716          0.000000  2563.346716
USD factor waterfall: {'carry': 14105.305887994778, 'rates_curves_pnl': 7263.838105378304, 'credit_curves_pnl': 2450.500893286646, 'inflation_curves_pnl': 0.0, 'correlations_pnl': 0.0, 'fx_pnl': 0.0, 'vol_pnl': 0.0, 'cross_factor_pnl': 0.0, 'model_params_pnl': 0.0, 'market_scalars_pnl': 3112.087777487934, 'residual': 0.0, 'reporting_fx_translation': -6023.149746946427}
USD bond Taylor residual, USD: -2034.686231345947
Taylor residual is disclosed approximation

## Required exercise: good allocation, poor selection


In [12]:
case=[{"sector":"Equity","portfolio_weight":.7,"benchmark_weight":.5,"portfolio_return":.07,"benchmark_return":.10},{"sector":"Bonds","portfolio_weight":.3,"benchmark_weight":.5,"portfolio_return":.01,"benchmark_return":.02}]
r=brinson_fachler(json.dumps(case))
assert r.total_allocation>0 and r.total_selection<0
assert abs(r.total_allocation+r.total_selection+r.total_interaction-r.total_excess_return)<1e-12
assert abs(r.total_allocation-.016)<1e-12
assert abs(r.total_selection+.02)<1e-12
print(r.to_json())
print("Overweight the stronger benchmark sector, yet underperform within both sectors; allocation is positive while selection is negative.")


{"sectors":[{"sector":"Equity","allocation":0.007999999999999998,"selection":-0.015,"interaction":-0.005999999999999998,"total":-0.013},{"sector":"Bonds","allocation":0.008000000000000002,"selection":-0.005,"interaction":0.002,"total":0.005000000000000002}],"total_allocation":0.016,"total_selection":-0.02,"total_interaction":-0.003999999999999998,"portfolio_return":0.052000000000000005,"benchmark_return":0.060000000000000005,"total_excess_return":-0.008}
Overweight the stronger benchmark sector, yet underperform within both sectors; allocation is positive while selection is negative.


## Required exercise: diagnose reconciliation and linking errors


In [13]:
restored = FiAttributionResult.from_json(attributed.to_json())
assert restored.reconciliation_check(1e-10).is_reconciled
assert abs(campisi_linked.active_return_compounded - (1.02**2 - 1)) < 1e-12
income_row=next(row for row in rows if row["receipts_usd"]>0)
missing_income=income_row["receipts_usd"]/income_row["opening_usd"]
assert missing_income>0
naive=sum(period.total_excess_return for period in brinson_periods)
print({"omitted_coupon_error_return":missing_income,"naive_active_sum":naive,"Carino_linked_active":active,"linking_difference":active-naive})
assert abs(linked.linked_allocation+linked.linked_selection+linked.linked_interaction-active)<1e-12


{'omitted_coupon_error_return': 0.02581654365984771, 'naive_active_sum': -0.0006155789219687196, 'Carino_linked_active': -0.0006125706468989023, 'linking_difference': 3.0082750698172747e-06}
